In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl) 

In [3]:
tabla_usuario = pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)
tabla_usuario.head(10)

,id,ciudad_id,user_id,telefono,area_id,cliente_id,sede_id,token_Firebase,lider
0,26,1,166,3317000000,167,11,34,NaN,False
1,31,1,169,3317000000,193,11,15,NaN,False
2,30,1,168,3317000000,193,11,16,NaN,False
3,32,1,176,3317000000,170,6,36,NaN,False
4,33,1,171,3317000000,168,6,36,NaN,False
5,34,1,172,3317000000,173,6,36,NaN,False
6,35,1,174,3317000000,169,6,36,NaN,False
7,36,1,175,3317000000,169,6,36,NaN,False
8,37,1,177,3317000000,171,6,36,NaN,False
9,39,1,179,3317000000,172,6,36,NaN,False


In [4]:
tabla_usuario_django = pd.read_sql_table('auth_user', mensajeria)
tabla_usuario_django['is_active'].value_counts()

is_active
True     327
False      2
Name: count, dtype: int64

In [5]:
tabla_areas=pd.read_sql_table('area',mensajeria)
tabla_areas.head(10)

,area_id,nombre,descripcion
0,6,CAPITALES,desc
1,8,LABORATORIO / CALI OCCIDENTE,desc
2,9,BANCO DE SANGRE / CAUCA OCCIDENTE,desc
3,13,RECEPCION / AAA,desc
4,14,FARMACIA /AAA,desc
5,15,REMEDIOS / CRUZ AZUL,desc
6,16,FARALLONES /AZUL,desc
7,17,PALMA REAL / CRUZMORADA,desc
8,18,DIME / ROJA,desc
9,19,DESPACHOS / ROJA,desc


In [6]:
dim_usuario = pd.merge(tabla_usuario,tabla_areas,left_on='area_id',right_on='area_id', how='left')
dim_usuario = dim_usuario.drop(columns=['telefono','sede_id','ciudad_id','token_Firebase','descripcion','cliente_id','lider','area_id'])
dim_usuario = pd.merge(dim_usuario,tabla_usuario_django,left_on='user_id',right_on='id',how='left')
dim_usuario.drop(columns=['password','last_login','is_superuser','email','is_staff','is_active','date_joined','user_id','id_y'],inplace=True)
dim_usuario.rename(columns={'id_x':'id_persona','username':'nombre_usuario','first_name':'nombre','last_name':'apellido','nombre':'area_trabajo'}, inplace=True)
dim_usuario['key_usuario'] = range(1,len(dim_usuario)+1)
dim_usuario.head(10)

,id_persona,area_trabajo,nombre_usuario,nombre,apellido,key_usuario
0,26,TALLER / PACIFICO,solarte7,pepito_el_rapido,pepito_el_furioso,1
1,31,REPUESTOS MOSTRADOR / PACIFICO,JGUTIERREZ,pepito_el_rapido,pepito_el_furioso,2
2,30,REPUESTOS MOSTRADOR / PACIFICO,ELOTERO,pepito_el_rapido,pepito_el_furioso,3
3,32,HOSPITALIZACION 3 PISO,HOSPITALIZACIONDIME,pepito_el_rapido,pepito_el_furioso,4
4,33,URGENCIAS,URGENCIADIME,pepito_el_rapido,pepito_el_furioso,5
5,34,CIRUGIA,CIRUGIADIME,pepito_el_rapido,pepito_el_furioso,6
6,35,UCI Y UCIN,UCIDIME,pepito_el_rapido,pepito_el_furioso,7
7,36,UCI Y UCIN,UCINDIME,pepito_el_rapido,pepito_el_furioso,8
8,37,INSUFICIENCIA CARDIACA,INSUFICIENCIADIME,pepito_el_rapido,pepito_el_furioso,9
9,39,HOSPITALIZACION 5 PISO,5PISODIME,pepito_el_rapido,pepito_el_furioso,10


In [7]:
dim_usuario.nunique()
dim_usuario.to_sql('dim_usuario', etl_conn, if_exists='replace', index=False) 

255